In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor

from sklearn.svm import SVR

import xgboost as xgb

In [7]:
df = pd.read_csv(r'D:\Emi Predict\Data Cleaning 1\emiPrediction_clean4.csv')

In [23]:
import pandas as pd

corr = df.corr(numeric_only=True)
target_corr = corr['max_monthly_emi'].sort_values(ascending=False)
print(target_corr)

max_monthly_emi           1.000000
emi_eligibility           0.539460
monthly_salary            0.460217
groceries_utilities       0.419788
travel_expenses           0.382440
bank_balance              0.343596
other_monthly_expenses    0.327501
house_type                0.325882
emergency_fund            0.324875
credit_score              0.253918
education                 0.197045
marital_status            0.085360
years_of_employment       0.020585
requested_tenure          0.001478
gender                    0.000585
company_type              0.000296
emi_scenario             -0.000277
requested_amount         -0.001289
age                      -0.017698
employment_type          -0.043172
school_fees              -0.309799
monthly_rent             -0.339444
college_fees             -0.386698
current_emi_amount       -0.388086
existing_loans           -0.402139
Name: max_monthly_emi, dtype: float64


In [8]:
from sklearn.model_selection import train_test_split

features = [
    "emi_eligibility","monthly_salary","current_emi_amount","college_fees","monthly_rent","school_fees",
    "groceries_utilities","travel_expenses","bank_balance","existing_loans"
] 

X = df[features]
y_regression = df['max_monthly_emi']
# Regression split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_regression, test_size=0.2, random_state=42)
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"Y_train: {y_train.shape}, Y_test: {y_test.shape}")

X_train: (320019, 10), X_test: (80005, 10)
Y_train: (320019,), Y_test: (80005,)


XGB Boost Regression

In [9]:
import xgboost as xgb
import pickle
xgb_model  = xgb.XGBRegressor(
        n_estimators=50,      # reduced from 200 → 50
        learning_rate=0.1,
        max_depth=3,          # smaller depth for speed
        random_state=42,
        n_jobs=-1             # parallel computation
    )
xgb_model.fit(X_train, y_train)
pickle.dump(xgb_model, open(r"D:\Emi Predict\Data Cleaning 1\reg_xgb1.pkl", "wb"))
print("Regression model saved")

Regression model saved


In [10]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

y_pred = xgb_model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

mape = np.mean(
    np.abs((y_test - y_pred) / np.maximum(y_test, 1))
) 

print("📊 Regression Metrics")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"R2   : {r2:.4f}")
print(f"MAPE : {mape:.2f}%")

📊 Regression Metrics
RMSE : 3211.5775
MAE  : 2143.1419
R2   : 0.8288
MAPE : 0.99%


Linear regression

In [11]:
import numpy as np
import pickle
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Pipeline (scaling + linear regression)
pipeline_linear = Pipeline([
    ('scaler', StandardScaler()),
    ('linreg', LinearRegression())
])

# Train model
pipeline_linear.fit(X_train, y_train)

# Predictions
y_pred_linear = pipeline_linear.predict(X_test)
# Evaluation metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred_linear))
mae = mean_absolute_error(y_test, y_pred_linear)
r2 = r2_score(y_test, y_pred_linear)

# Avoid division by zero in MAPE
mape = np.mean(np.abs((y_test - y_pred_linear) / (y_test + 1e-8))) * 100

print("📊 Regression Metrics")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"R2   : {r2:.4f}")
print(f"MAPE : {mape:.2f}%")

# Save model
pickle.dump(pipeline_linear, open(r"D:\Emi Predict\Data Cleaning 1\reg_linear1.pkl", "wb"))

📊 Regression Metrics
RMSE : 3755.7353
MAE  : 2556.5428
R2   : 0.7658
MAPE : 155.73%


In [ ]:
import numpy as np
import pickle
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Pipeline
pipeline_lin = Pipeline([
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(include_bias=False)),
    ('model', Ridge())   # placeholder (will be replaced in grid)
])

# Parameter grid (Ridge, Lasso, ElasticNet)
param_grid = [
    {
        'poly__degree': [1, 2, 3],
        'model': [Ridge()],
        'model__alpha': [0.001, 0.01, 0.1, 1, 10, 100]
    },
    {
        'poly__degree': [1, 2, 3],
        'model': [Lasso(max_iter=5000)],
        'model__alpha': [0.001, 0.01, 0.1, 1, 10]
    },
    {
        'poly__degree': [1, 2, 3],
        'model': [ElasticNet(max_iter=5000)],
        'model__alpha': [0.001, 0.01, 0.1, 1, 10],
        'model__l1_ratio': [0.1, 0.5, 0.9]
    }
]

# Cross-validation
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# Grid Search
grid = GridSearchCV(
    pipeline_lin,
    param_grid,
    cv=cv,
    scoring='r2',   # optimize for R²
    n_jobs=-1,
    verbose=1
)

# Train
grid.fit(X_train, y_train)

# Best model
best_model = grid.best_estimator_

# Predictions
y_pred_lin1 = best_model.predict(X_test)

# Metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred_lin1))
mae = mean_absolute_error(y_test, y_pred_lin1)
r2 = r2_score(y_test, y_pred_lin1)
mape = np.mean(np.abs((y_test - y_pred_lin1) / (y_test + 1e-8))) * 100

print("Best Parameters:", grid.best_params_)

print("\n📊 Regression Metrics")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"R2   : {r2:.4f}")
print(f"MAPE : {mape:.2f}%")

# Save model
pickle.dump(best_model, open(r"D:\Emi Predict\Data Cleaning 1\reg_linear_tuned.pkl", "wb"))

Fitting 5 folds for each of 78 candidates, totalling 390 fits


RANDOM FOREST REGRESSOR

In [ ]:
import numpy as np
import pickle
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Pipeline (scaling + random forest)
pipeline_RF = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestRegressor(random_state=42))
])

# Train model
pipeline_RF.fit(X_train, y_train)

# Predictions
y_pred_RF = pipeline_RF.predict(X_test)

# Evaluation metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred_RF))
mae = mean_absolute_error(y_test, y_pred_RF)
r2 = r2_score(y_test, y_pred_RF)

# Avoid division by zero in MAPE
mape = np.mean(np.abs((y_test - y_pred_RF) / (y_test + 1e-8))) * 100

print("📊 Regression Metrics")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"R2   : {r2:.4f}")
print(f"MAPE : {mape:.2f}%")

# Save model
with open(r"D:\Emi Predict\Data Cleaning 1\reg_RF1.pkl", "wb") as f:
    pickle.dump(pipeline_RF, f)

📊 Regression Metrics
RMSE : 2419.9456
MAE  : 1383.1361
R2   : 0.9028
MAPE : 30.80%


In [ ]:
import numpy as np
import pickle
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Pipeline (RF doesn't need scaling, but keeping structure consistent)
pipeline_rf = Pipeline([
    ('rf', RandomForestRegressor(random_state=42))
])

# Hyperparameter distribution
param_dist = {
    'rf__n_estimators': [100, 200, 300, 500],
    'rf__max_depth': [None, 10, 20, 30, 50],
    'rf__min_samples_split': [2, 5, 10],
    'rf__min_samples_leaf': [1, 2, 4],
    'rf__max_features': ['sqrt', 'log2', None],
    'rf__bootstrap': [True, False]
}

# Cross-validation
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# Randomized Search (faster than GridSearch)
search = RandomizedSearchCV(
    pipeline_rf,
    param_distributions=param_dist,
    n_iter=50,
    cv=cv,
    scoring='r2',
    n_jobs=-1,
    verbose=1,
    random_state=42
)

# Train
search.fit(X_train, y_train)

# Best model
rf_model = search.best_estimator_

# Predictions
y_pred_rf = rf_model.predict(X_test)

# Metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))
mae = mean_absolute_error(y_test, y_pred_rf)
r2 = r2_score(y_test, y_pred_rf)
mape = np.mean(np.abs((y_test - y_pred_rf) / (y_test + 1e-8))) * 100

print("Best Parameters:", search.best_params_)

print("\n📊 Regression Metrics")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"R2   : {r2:.4f}")
print(f"MAPE : {mape:.2f}%")

# Save model
pickle.dump(rf_model, open(r"D:\Emi Predict\Data Cleaning 1\reg_rf_tuned.pkl", "wb"))

In [ ]:
BEST MODEL

In [78]:
def evaluate_regression(y_true, y_pred):

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    return rmse, mae, r2, mape

In [79]:
models = {
    "Linear Regression": pipeline_linear,
    "Random Forest": pipeline_RF,
    "XGBoost": xgb_model
}

results = []

for name, model in models.items():
    print(f"Training {name}...")
    
    # Train model
    model.fit(X_train, y_train)
    
    # Predict
    preds = model.predict(X_test)
    
    # Evaluate
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    
    results.append({
        "Model": name,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    })

# Convert results to DataFrame
results_df = pd.DataFrame(results)

print("\nModel Performance:")
print(results_df)

Training Linear Regression...
Training Random Forest...
Training XGBoost...

Model Performance:
               Model      RMSE       MAE        R2
0  Linear Regression  0.655346  0.511744  0.769275
1      Random Forest  0.345317  0.228576  0.935939
2            XGBoost  0.570234  0.443575  0.825314


In [17]:
results_df = results_df.sort_values(by="R2", ascending=False)
print("Model Comparison:")
print(results_df)

Model Comparison:
               Model          RMSE           MAE        R2
0  Linear Regression  1.964879e-16  1.409446e-16  1.000000
1      Random Forest  1.284740e-05  3.257877e-07  1.000000
2            XGBoost  3.900489e-04  1.380906e-04  0.999936


In [18]:
best_model_name = results_df.iloc[0]["Model"]

print("Best Regression Model:", best_model_name)

Best Regression Model: Linear Regression


In [12]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Initialize and train Ridge Regression
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)

# Predictions
y_pred_ridge = ridge.predict(X_test)

# Evaluation
mae = mean_absolute_error(y_test, y_pred_ridge)
mse = mean_squared_error(y_test, y_pred_ridge)
rmse = np.sqrt(mse)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred_ridge)

print(f"Ridge Regression Results:")
print(f"MAE  : {mae:.4f}")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")

Ridge Regression Results:
MAE  : 0.5117
MSE  : 0.4295
RMSE : 0.6553
R²   : 0.7693
